# 卷积神经网络（CNN）

本 notebook 分两部分：

- **上篇 · 知识点**：先把 CNN 的核心概念讲清楚——为什么要用卷积、卷积 / 池化怎么算、网络怎么搭、层数怎么定
- **下篇 · 实践**：再动手跑两个完整示例，流程一模一样，只有 3 处参数不同

| 示例 | 数据集 | 图片 | 通道 | 脚本 |
|---|---|---|---|---|
| 手写数字识别 | MNIST | 28x28 灰度 | 1 | `cnn_number.py` |
| 彩色图片分类 | CIFAR-10 | 32x32 RGB | 3 | `cnn_cifar10.py` |

> 数据由 `prepare_data.py` 预先下载并转成 npz，存放在 `09_cnn/data/` 下。


## 上篇 · 知识点

### 一、为什么图片任务要用 CNN，而不是全连接网络？

全连接网络（07/08 章）必须先把图片 Flatten 成一维向量（28x28 = 784）再送进去，
这一步会把「上下左右相邻像素」的空间关系彻底丢掉——而笔画、边缘、拐角这些图案，
恰恰靠空间位置关系才能识别。

卷积层（`Conv2D`）用一个小卷积核（如 3x3）在整张图上滑动，有两个关键好处：

1. **局部连接**：每个输出只依赖一小块区域，天然适合捕捉局部笔画 / 边缘
2. **权值共享**：同一个卷积核扫遍全图，参数量远小于全连接

参数量对比（都提取 32 个特征图）：

| 方式 | 参数计算 | 参数量 |
|---|---|---|
| 3x3 卷积核 × 32 | 3×3×1×32 + 32 | **320** |
| 784 维输入 → 32 个神经元 | 784×32 + 32 | 25088 |

卷积把参数量压到约 1/78，同时还保住了空间结构。


### 二、卷积运算原理

**卷积核（滤波器 / kernel / filter）** 是一个小矩阵（如 3x3）。它盖在图片的一小块区域上，
对应位置相乘再求和，得到一个数；然后按**步长（stride）** 滑动到下一位置，重复计算。
一次滑动产生的所有数拼成一张**特征图（feature map）**。

```
输入局部 3x3             卷积核 3x3            输出
[1 0 1]
[0 1 0]      ⊛        [1 0 1]      =      对应位置相乘再求和 → 一个数
[1 1 0]               [0 1 0]
                      [1 0 1]
                     （沿整张图滑动，所有结果拼成一张特征图）
```

几个关键概念：

| 概念 | 含义 |
|---|---|
| 卷积核大小 `kernel_size` | 决定一次看多大范围，`3x3` 最常用 |
| 步长 `stride` | 每次滑动几格，默认 1；步长 2 则尺寸减半 |
| 填充 `padding` | 图像边缘补 0；`same` 让输出尺寸与输入一致，`valid` 不补 |
| `filters` | 卷积核的个数 = 输出通道数 = 提取多少种特征 |
| 感受野 | 深层神经元对应到原图上的区域大小，层数越深感受野越大 |

**输出尺寸公式**（输入高 `H`、核 `K`、填充 `P`、步长 `S`）：

$$
H_{out} = \left\lfloor \frac{H - K + 2P}{S} \right\rfloor + 1
$$

- 不带 padding、步长 1：`输出 = H - K + 1`（如 28 -> 26）
- `padding='same'`、步长 1：`输出 = H`（尺寸不变）

> 一个卷积核只能学一种特征（如某个方向的边缘）。要同时学「横边缘、竖边缘、纹理、颜色斑点……」，
> 就用多个卷积核并排——这就是 `filters=32` 的含义：输出 32 张特征图。


### 三、激活函数：ReLU

卷积之后要接一个非线性激活，把负值置 0、正值保留：

$$
\text{ReLU}(z) = \max(0,\, z)
$$

- 作用：引入非线性。若全部是线性层叠加，等价于一层线性变换，加深网络没有意义
- 相比 sigmoid，ReLU 计算快，且两端不像 sigmoid 那样梯度趋近 0（缓解梯度消失）
- 代价：可能出现「死亡」神经元（输入恒为负则该神经元永远输出 0）


### 四、池化层：降采样

`MaxPooling2D(2, 2)` 在每 2x2 区域取**最大值**，宽高各减半。作用：

1. **降采样**：缩小特征图，减少后续计算量和参数量
2. **平移不变性**：目标稍微偏移，池化后的值基本不变，识别更稳
3. **扩大感受野**：同样的核用在缩小后的图上，等价于看更大范围

另一种是平均池化（取平均），实际最常用的还是最大池化。
池化层**没有可训练参数**——它只是取最大值，不做加权。


### 五、通道与张量维度的变化

一张图的张量形状是 `(高, 宽, 通道数)`，进入网络后逐层变化，以 MNIST 为例：

| 层 | 形状 | 说明 |
|---|---|---|
| 输入 | (28, 28, 1) | 灰度图通道数 = 1 |
| Conv2D(32, 3x3) | (26, 26, 32) | 32 个卷积核 → 32 个通道 |
| MaxPooling(2x2) | (13, 13, 32) | 宽高减半，通道不变 |
| Conv2D(64, 3x3) | (11, 11, 64) | 通道 32 -> 64 |
| MaxPooling(2x2) | (5, 5, 64) | |
| Conv2D(64, 3x3) | (3, 3, 64) | |
| Flatten | (576,) | 3×3×64 = 576 |

**规律**：卷积改变通道数（由 `filters` 决定），池化只改变宽高、通道不变；
尺寸减半时通常让通道翻倍（32 -> 64），保持信息量平衡。
浅层学局部细节（边缘 / 笔画），深层学更抽象的组合特征。


### 六、分类头：从特征图到类别

卷积部分得到的是特征图，最后要靠**分类头**把它变成各类别的概率：

```
Flatten -> Dropout -> Dense(relu) -> Dense(softmax)
```

- `Flatten`：把 (高, 宽, 通道) 三维特征图拉平成一维向量。`Dense` 只接受一维输入，它是卷积与全连接之间的桥
- `Dropout(0.5)`：训练时随机关掉一半神经元，防止过拟合（只在训练时生效，预测 / 评估时自动关闭）
- `Dense(relu)`：把卷积提取的特征组合起来做非线性变换
- `Dense(类别数, softmax)`：输出每个类别的概率（各项在 0~1、总和为 1），取最大值的下标即预测类别


### 七、层数怎么定？——「尺寸倒推法」

每做一次 `MaxPooling2D(2x2)`，特征图宽高就减半，所以层数由**输入尺寸**决定，不是拍脑袋：

- **MNIST（28x28）**：28 -> 14 -> 7 -> 3，最多池化 3 次
  （再池化就变 1x1，空间信息全没了，卷积失去意义）=> 3 个卷积块
- **CIFAR-10（32x32）**：32 -> 16 -> 8 -> 4 -> 2，尺寸宽裕，用 3 个卷积块（池化 3 次落到 4x4）

> 作为对比：VGG16 吃 224x224 的大图，能池化 5~6 次，所以能堆到 16 层；
> 拿 28x28 的 MNIST 去堆 50 层，池化 5 次就到 1x1，纯属浪费。


### 八、CNN 整体结构与组件速查

一个典型 CNN 由「卷积块 × N + 分类头」组成：

```
卷积块：  Conv2D -> ReLU -> MaxPooling2D       （重复若干次）
分类头：  Flatten -> Dropout -> Dense -> Dense(softmax)
```

| 组件 | 作用 | 可训练参数 | 改变什么 |
|---|---|---|---|
| Conv2D | 提取局部特征 | 有 | 通道数（由 filters 定）、宽高（由 padding / stride 定） |
| MaxPooling2D | 降采样、平移不变性 | 无 | 宽高减半 |
| ReLU | 引入非线性 | 无 | 不变 |
| Flatten | 三维 -> 一维 | 无 | 形状 |
| Dropout | 防过拟合 | 无 | 训练时随机置零 |
| Dense | 特征组合 / 分类 | 有 | 维度 |

**训练三步**：编译（optimizer / loss / metrics）→ `fit`（含 `validation_split`）→ `evaluate` / `predict`。

知识点讲完了，下面进入实践部分。


## 下篇 · 实践

下面两个例子流程完全一致：**数据准备 → 搭建模型 → 编译 → 训练 → 评估预测 → 可视化**，
差别只在 3 个地方（见文末「示例对比」）。

### 数据准备

`prepare_data.py` 负责从 `tf.keras.datasets` 下载数据并转成压缩 npz，存到 `data/` 下：

| 数据集 | 命令 | 输出文件 | x 形状 |
|---|---|---|---|
| MNIST | `python prepare_data.py` | `data/train.npz` / `data/test.npz` | (60000, 28, 28) |
| CIFAR-10 | `python prepare_data.py cifar10` | `data/cifar10_train.npz` / `cifar10_test.npz` | (50000, 32, 32, 3) |

- npz 里存**原始 0~255 的 uint8 像素**，不做归一化；归一化放到训练脚本里，方便自由调整预处理方式
- 首次运行联网下载并缓存到 `~/.keras/datasets`，之后直接读缓存，不再联网


In [ ]:
%matplotlib inline

import os

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

# 让 matplotlib 正常显示中文和负号
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 定位数据目录：兼容从仓库根目录或 09_cnn 目录启动 notebook
for cand in ['09_cnn/data', 'data', '../data']:
    if os.path.isdir(cand):
        DATA_DIR = cand
        break
print('DATA_DIR =', DATA_DIR)
print('TensorFlow', tf.__version__)


### 例 1：手写数字识别（MNIST，灰度图）

网络结构：
`Conv2D(32) -> Pool -> Conv2D(64) -> Pool -> Conv2D(64) -> Flatten -> Dropout -> Dense(64) -> Dense(10)`

尺寸变化（不带 padding）：28 -> 26 -> 13 -> 11 -> 5 -> 3


In [ ]:
# ---------- MNIST 数据准备 ----------
train_data = np.load(os.path.join(DATA_DIR, 'train.npz'))
test_data = np.load(os.path.join(DATA_DIR, 'test.npz'))

# npz 里是原始 uint8 像素(0~255)，形状 (N, 28, 28)
x_train, y_train = train_data['x'], train_data['y']
x_test, y_test = test_data['x'], test_data['y']

class_names = [str(i) for i in range(10)]

# 归一化：0~255 -> 0~1（神经网络对输入数值范围敏感）
x_train = (x_train / 255.0).astype('float32')
x_test = (x_test / 255.0).astype('float32')

# 加通道维：Conv2D 要求 (N, 高, 宽, 通道数)，灰度图通道数 = 1
x_train = x_train[..., np.newaxis]   # (N,28,28) -> (N,28,28,1)
x_test = x_test[..., np.newaxis]

print(f'训练集：x={x_train.shape}, y={y_train.shape}')
print(f'测试集：x={x_test.shape}, y={y_test.shape}')
print(f'像素范围：{x_train.min()} ~ {x_train.max()}')


In [ ]:
# ---------- MNIST 模型 ----------
model = models.Sequential()

# Keras 3 推荐用独立 Input 层声明输入形状 (高, 宽, 通道数)，只有第一层需要写
model.add(layers.Input(shape=(28, 28, 1)))

# 卷积块 1
model.add(layers.Conv2D(32, (3, 3), activation='relu'))   # 28-3+1 = 26
model.add(layers.MaxPooling2D((2, 2)))                    # 26 -> 13

# 卷积块 2：通道 32 -> 64 翻倍
model.add(layers.Conv2D(64, (3, 3), activation='relu'))   # 13-3+1 = 11
model.add(layers.MaxPooling2D((2, 2)))                    # 11 -> 5

# 卷积块 3：后面不接池化，尺寸停在 3x3
# 用卷积把 (5,5,64)=1600 维压到 (3,3,64)=576 维，后面 Dense 能省一大半参数
model.add(layers.Conv2D(64, (3, 3), activation='relu'))   # 5-3+1 = 3

# 分类头
model.add(layers.Flatten())                               # (3,3,64) -> 576
model.add(layers.Dropout(0.5))                            # 防过拟合
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))         # 10 类

model.summary()


In [ ]:
model.compile(
    # Adam 自适应学习率，收敛快，一般默认就用它
    optimizer='adam',
    # 多分类 + 整数标签(0~9) -> sparse_categorical_crossentropy，配 softmax 输出层
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# fit 循环 epochs 次：前向传播 -> 算损失 -> 反向传播 -> 更新权重
history = model.fit(
    x_train, y_train,
    batch_size=32,
    epochs=5,               # MNIST 简单，5 轮就能到 99% 左右
    validation_split=0.1,   # 切 10% 做验证集；测试集只在最后评估用一次
    verbose=1,
)


In [ ]:
# evaluate：在测试集上算损失和准确率（不做参数更新）
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'\n测试集结果：loss={test_loss:.4f}，accuracy={test_acc:.4f}')

# predict：得到每个样本属于 10 个类的概率，形状 (N, 10)
probs = model.predict(x_test, verbose=0)
y_pred = probs.argmax(axis=1)          # 取概率最大的类别

print('\n各类别测试准确率：')
for i, name in enumerate(class_names):
    mask = y_test == i                 # 选出真实标签为 i 的样本
    acc = (y_pred[mask] == y_test[mask]).mean()
    print(f'  数字 {name}：{acc:.4f}（共 {mask.sum()} 张）')


In [ ]:
# 图 1：训练曲线
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='训练 loss')
plt.plot(history.history['val_loss'], label='验证 loss')
plt.plot(history.history['accuracy'], label='训练 accuracy')
plt.plot(history.history['val_accuracy'], label='验证 accuracy')
plt.xlabel('epoch'); plt.ylabel('value'); plt.title('训练曲线')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# 图 2：随机抽 10 张测试图，显示预测结果
idx = np.random.choice(len(x_test), 10, replace=False)
plt.figure(figsize=(14, 6))
for i, k in enumerate(idx):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[k].reshape(28, 28), cmap='gray')
    correct = y_pred[k] == y_test[k]   # 预测对绿色，预测错红色
    plt.title(f'预测:{class_names[y_pred[k]]} 真实:{class_names[y_test[k]]}',
              color='green' if correct else 'red')
    plt.axis('off')
plt.tight_layout(); plt.show()


### 例 2：彩色图片分类（CIFAR-10，RGB 三通道）

数据集：32x32 彩色图，10 类——airplane / automobile / bird / cat / deer / dog / frog / horse / ship / truck。

**和 MNIST 例子的流程完全一样，只有 3 处不同：**

| # | 差异点 | MNIST（灰度） | CIFAR-10（彩色） |
|---|---|---|---|
| 1 | 输入通道数 | 1，`Input(shape=(28,28,1))` | 3，`Input(shape=(32,32,3))` |
| 2 | 是否手动加通道维 | 需 `[..., np.newaxis]` 补成 (N,28,28,1) | 数据本身是 (N,32,32,3)，不用加 |
| 3 | 第一层卷积核深度 | 3x3x1，32 核 = 320 参数 | 3x3x3，32 核 = 896 参数 |

> 第 3 点**只有第一层受影响**：第二层往后的输入通道数由上一层的 `filters` 决定，
> 跟原图有几个通道无关，所以代码写法完全一致。
>
> 为什么流程能一样？`Conv2D` 做的是三维卷积（高、宽、通道），对通道方向做加权求和后压成一个数，
> 输出永远是一张二维特征图。灰度是「1 个通道求和」，彩色是「3 个通道求和」，机制没有区别。
>
> 注意：CIFAR-10 比 MNIST 难得多（彩色、目标形态多变、类间相似如猫狗），**准确率 70% 左右就是正常水平**，
> 不要拿 MNIST 的 99% 来对比。


In [ ]:
# ---------- CIFAR-10 数据准备 ----------
cifar_train = np.load(os.path.join(DATA_DIR, 'cifar10_train.npz'))
cifar_test = np.load(os.path.join(DATA_DIR, 'cifar10_test.npz'))

# 形状已经是 (N, 32, 32, 3)，符合 Conv2D 要求，不用加通道维
cx_train, cy_train = cifar_train['x'], cifar_train['y']
cx_test, cy_test = cifar_test['x'], cifar_test['y']

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

cx_train = (cx_train / 255.0).astype('float32')
cx_test = (cx_test / 255.0).astype('float32')

print(f'训练集：x={cx_train.shape}, y={cy_train.shape}')
print(f'测试集：x={cx_test.shape}, y={cy_test.shape}')
print(f'像素范围：{cx_train.min()} ~ {cx_train.max()}')


In [ ]:
# ---------- CIFAR-10 模型 ----------
cmodel = models.Sequential()
cmodel.add(layers.Input(shape=(32, 32, 3)))              # 彩色图通道数 = 3

# padding='same'：边缘补 0，输出尺寸先保持，再由池化减半
cmodel.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))  # 32
cmodel.add(layers.MaxPooling2D((2, 2)))                                   # 32 -> 16

cmodel.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))  # 16
cmodel.add(layers.MaxPooling2D((2, 2)))                                   # 16 -> 8

cmodel.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))  # 8
cmodel.add(layers.MaxPooling2D((2, 2)))                                   # 8 -> 4

cmodel.add(layers.Flatten())                             # (4,4,64) -> 1024
cmodel.add(layers.Dropout(0.5))
cmodel.add(layers.Dense(128, activation='relu'))
cmodel.add(layers.Dense(10, activation='softmax'))

cmodel.summary()


In [ ]:
cmodel.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# CIFAR-10 比 MNIST 难得多，轮数要给够
# （实测 10 轮时 loss 还在下降，所以这里用 30 轮；彩色图更占内存，batch_size 调大一点）
chistory = cmodel.fit(
    cx_train, cy_train,
    batch_size=64,
    epochs=30,
    validation_split=0.1,
    verbose=1,
)


In [ ]:
ctest_loss, ctest_acc = cmodel.evaluate(cx_test, cy_test, verbose=0)
print(f'\n测试集结果：loss={ctest_loss:.4f}，accuracy={ctest_acc:.4f}')

cprobs = cmodel.predict(cx_test, verbose=0)
cy_pred = cprobs.argmax(axis=1)

# 逐类看准确率，能发现「哪类最容易认错」（比如猫狗常常互相认错）
print('\n各类别测试准确率：')
for i, name in enumerate(class_names):
    mask = cy_test == i
    acc = (cy_pred[mask] == cy_test[mask]).mean()
    print(f'  {name:<11}：{acc:.4f}（共 {mask.sum()} 张）')


In [ ]:
# 图 1：训练曲线
plt.figure(figsize=(8, 5))
plt.plot(chistory.history['loss'], label='训练 loss')
plt.plot(chistory.history['val_loss'], label='验证 loss')
plt.plot(chistory.history['accuracy'], label='训练 accuracy')
plt.plot(chistory.history['val_accuracy'], label='验证 accuracy')
plt.xlabel('epoch'); plt.ylabel('value'); plt.title('训练曲线')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# 图 2：随机抽 10 张测试图
idx = np.random.choice(len(cx_test), 10, replace=False)
plt.figure(figsize=(14, 6))
for i, k in enumerate(idx):
    plt.subplot(2, 5, i + 1)
    plt.imshow(cx_test[k])          # 彩色图直接 imshow，不加 cmap，也不用 reshape
    correct = cy_pred[k] == cy_test[k]
    plt.title(f'预测:{class_names[cy_pred[k]]}\n真实:{class_names[cy_test[k]]}',
              color='green' if correct else 'red')
    plt.axis('off')
plt.tight_layout(); plt.show()


## 示例对比

| 维度 | MNIST（`cnn_number.py`） | CIFAR-10（`cnn_cifar10.py`） |
|---|---|---|
| 图片 | 28x28 灰度 | 32x32 RGB |
| 输入通道 | 1 | 3 |
| 需手动加通道维 | 要 `[..., np.newaxis]` | 不要 |
| 第一层卷积核 | 3x3x1 -> 32 个，320 参数 | 3x3x3 -> 32 个，896 参数 |
| 池化次数 | 3 次（28 -> 14 -> 7 -> 3） | 3 次（32 -> 16 -> 8 -> 4） |
| padding | 不带（每层缩 2） | `same`（尺寸保持后减半） |
| Flatten 后维度 | (3,3,64) = 576 | (4,4,64) = 1024 |
| batch_size / epochs | 32 / 5 | 64 / 30 |
| 正常准确率 | ~99% | ~70% |


## 小结

```
CNN 结构   卷积块 × N（Conv2D -> ReLU -> MaxPooling2D） + 分类头（Flatten -> Dropout -> Dense）
卷积       局部连接 + 权值共享，捕捉空间特征，参数量远小于全连接
池化       每 2x2 取最大，宽高减半；降采样 + 平移不变性
层数定法   尺寸倒推法：每次池化减半，别让尺寸落到 1x1
通道规律   尺寸减半 -> 通道翻倍（32 -> 64）
输出层     多分类用 softmax（10 类），损失 sparse_categorical_crossentropy
训练三步   编译（optimizer/loss/metrics） -> fit（含 validation_split） -> evaluate / predict
防过拟合   Dropout(0.5) + 用验证集监控训练曲线
```

- 归一化（/255.0）必须在训练前做，否则梯度尺度混乱，训练又慢又不稳定
- 测试集只在最后评估用一次，不要参与训练或调参（用 `validation_split` 从训练集切验证集）
- 数据形状：灰度 (N,H,W) 需补通道维；彩色 (N,H,W,3) 直接可用
